In [3]:
!pip install pandas numpy faiss-cpu sentence-transformers tqdm flask scikit-learn


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googletrans 4.0.0rc1 requires httpx==0.13.3, but you have httpx 0.28.1 which is incompatible.
gtts 2.5.1 requires click<8.2,>=7.1, but you have click 8.3.3 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
  Using cached transformers-5.7.0-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.13.0-py3-none-any.whl.metadata (14 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-an

# 1. Import Libraries

In [4]:
import pandas as pd
import numpy as np
import re
import pickle
import os
from sentence_transformers import SentenceTransformer
import faiss
from tqdm import tqdm


c:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Load Dataset
#### Load the Hadith dataset

In [5]:
df = pd.read_csv('all_hadiths_clean.csv')

print(f"Original dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Original dataset shape: (34441, 9)
Columns: ['id', 'hadith_id', 'source', 'chapter_no', 'hadith_no', 'chapter', 'chain_indx', 'text_ar', 'text_en']


,id,hadith_id,source,chapter_no,hadith_no,chapter,chain_indx,text_ar,text_en
0,0,1,Sahih Bukhari,1,1,Revelation - كتاب بدء الوحى,"30418, 20005, 11062, 11213, 11042, 3",حدثنا الحميدي عبد الله بن الزبير، قال حدثنا سف...,Narrated 'Umar bin Al-Khattab: ...
1,1,2,Sahih Bukhari,1,2,Revelation - كتاب بدء الوحى,"30355, 20001, 11065, 10511, 53",حدثنا عبد الله بن يوسف، قال أخبرنا مالك، عن هش...,Narrated 'Aisha: ...
2,2,3,Sahih Bukhari,1,3,Revelation - كتاب بدء الوحى,"30399, 20023, 11207, 11013, 10511, 53",حدثنا يحيى بن بكير، قال حدثنا الليث، عن عقيل، ...,Narrated 'Aisha: (the m...
3,3,4,Sahih Bukhari,1,4,Revelation - كتاب بدء الوحى,"11013, 10567, 34",قال ابن شهاب وأخبرني أبو سلمة بن عبد الرحمن، أ...,Narrated Jabir bin 'Abdullah Al-Ansari while ...
4,4,5,Sahih Bukhari,1,5,Revelation - كتاب بدء الوحى,"20040, 20469, 11399, 11050, 17",حدثنا موسى بن إسماعيل، قال حدثنا أبو عوانة، قا...,Narrated Said bin Jubair: ...


# 4. Data Preprocessing

In [6]:
def preprocess_text(text):
    """Clean and normalize text"""
    if pd.isna(text):
        return ""
    
    # Convert to string
    text = str(text)
    
    # Lowercase
    text = text.lower()
    
    # Remove special characters (keep Arabic characters)
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

#### Create a combined text field for better search

In [7]:
df['search_text'] = df.apply(lambda row: f"{row['chapter']} {row['text_en']}", axis=1)

# Remove null or empty entries
df = df.dropna(subset=['text_en', 'chapter'])
df = df[df['text_en'].str.strip() != '']
df = df[df['chapter'].str.strip() != '']

# Preprocess text
df['clean_text'] = df['search_text'].apply(preprocess_text)

# Remove duplicates based on clean text
df = df.drop_duplicates(subset=['clean_text'])

print(f"Cleaned dataset shape: {df.shape}")
print(f"Sample clean text:\n{df['clean_text'].iloc[0][:200]}")

Cleaned dataset shape: (33475, 11)
Sample clean text:
revelation كتاب بدء الوحى narrated umar bin al khattab i heard allah s apostle saying the reward of deeds depends upon the intentions and every person will get the reward according to what he has inte


# 5. Save Cleaned Dataset

In [8]:
os.makedirs('data', exist_ok=True)

# Save cleaned dataset
df_cleaned = df[['hadith_id', 'source', 'chapter', 'hadith_no', 'text_en', 'clean_text']]
df_cleaned.to_csv('data/cleaned_hadiths.csv', index=False)
print("Cleaned dataset saved to data/cleaned_hadiths.csv")

Cleaned dataset saved to data/cleaned_hadiths.csv


# 6. Generate Embeddings
### Load the MiniLM model

In [9]:
print("Loading sentence transformer model...")
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Generate embeddings for all Hadith texts
print("Generating embeddings...")
texts = df['clean_text'].tolist()

# Generate embeddings in batches
batch_size = 32
embeddings = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_embeddings = model.encode(batch, show_progress_bar=False)
    embeddings.append(batch_embeddings)

embeddings = np.vstack(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

Loading sentence transformer model...


c:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7190.51it/s]


Generating embeddings...


100%|██████████| 1047/1047 [33:33<00:00,  1.92s/it]


Embeddings shape: (33475, 384)


# 7. Create FAISS Index

In [10]:
# Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner product for cosine similarity
index.add(embeddings)

print(f"FAISS index created with {index.ntotal} vectors")

FAISS index created with 33475 vectors


# 8. Save FAISS Index and Metadata

In [11]:
# Create faiss_index directory if it doesn't exist
os.makedirs('faiss_index', exist_ok=True)

# Save FAISS index
faiss.write_index(index, 'faiss_index/hadith_index.faiss')
print("FAISS index saved to faiss_index/hadith_index.faiss")

# Save metadata (for retrieving Hadith details)
metadata = {
    'hadith_ids': df['hadith_id'].tolist(),
    'sources': df['source'].tolist(),
    'chapters': df['chapter'].tolist(),
    'hadith_nos': df['hadith_no'].tolist(),
    'texts': df['text_en'].tolist(),
    'indices': list(range(len(df)))
}

with open('faiss_index/metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print("Metadata saved to faiss_index/metadata.pkl")

# Save the mapping DataFrame
df[['hadith_id', 'source', 'chapter', 'hadith_no', 'text_en']].to_csv('data/hadith_mapping.csv', index=False)
print("Hadith mapping saved to data/hadith_mapping.csv")

FAISS index saved to faiss_index/hadith_index.faiss
Metadata saved to faiss_index/metadata.pkl
Hadith mapping saved to data/hadith_mapping.csv


# 9. Test the Index

In [12]:
# Test search
test_query = "what are the five pillars of islam"
query_embedding = model.encode([test_query])
faiss.normalize_L2(query_embedding)

distances, indices = index.search(query_embedding, 3)

print(f"\nTest Query: '{test_query}'")
print("\nTop 3 Results:")
for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"\n--- Result {i+1} (Score: {dist:.4f}) ---")
    print(f"Source: {df.iloc[idx]['source']}")
    print(f"Chapter: {df.iloc[idx]['chapter']}")
    print(f"Hadith: {df.iloc[idx]['text_en'][:200]}...")

# %%
print("\n Preprocessing Complete!")
print("FAISS index is ready for the Flask application.")
print("\nNext step: Run 'python app.py' to start the Flask server.")


Test Query: 'what are the five pillars of islam'

Top 3 Results:

--- Result 1 (Score: 0.6463) ---
Source:  Sahih Muslim 
Chapter: The Book of Faith - كتاب الإيمان
Hadith:  It is narrated on the authority of 'Abdullah son of 'Umar that the Messenger of Allah (may peace be upon him) said:                      (The superstructure of) al-Islam is raised on five (pillars), ...

--- Result 2 (Score: 0.6391) ---
Source:  Sunan an-Nasa'i 
Chapter: The Book Of Faith and its Signs - كتاب الإيمان وشرائعه
Hadith:  It was narrated from Ibn 'Umar that:                     A man said to him: "Why don't you go out and fight?" He said: "I heard the Messenger of Allah [SAW] say: 'Islam is built on five (pillars): Te...

--- Result 3 (Score: 0.6389) ---
Source:  Sahih Muslim 
Chapter: The Book of Faith - كتاب الإيمان
Hadith:  It is narrated on the authority of ('Abdullah) son of 'Umar, that the Holy Prophet (may peace of Allah be upon him) said:                      (The superstructure of) al-Islam is 